# Быстрый старт: панель, показатели, сеть и кластеры в коде

Ноутбук для аналитиков команды: как загрузить готовые данные и построить сеть и кластеры теми же функциями, что
использует интерфейс (`streamlit run app/Home.py`). Перед запуском соберите данные: `python scripts/run_all.py`.

In [1]:
import pathlib
import sys

# корень репозитория в пути импорта (ноутбук можно запускать из notebooks/ или из корня)
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd  # noqa: E402

from src import clustering, convergence, dynamics, icvi, network  # noqa: E402
from src.io import PROCESSED  # noqa: E402

## 1. Данные
- `panel_long.parquet` — исходные ряды (МО × год × ряд) с флагами качества;
- `indicators_wide.parquet` — показатели из реестра `configs/indicators.yaml` (одна строка на МО × год);
- `mo.parquet` — справочник МО (название, регион, ФО, координаты, смены границ).

Для **среза по году** берите строки `valid_in_year == True`.

In [2]:
mo = pd.read_parquet(PROCESSED / "mo.parquet")
ind = pd.read_parquet(PROCESSED / "indicators_wide.parquet")
dfo22 = ind[ind.is_dfo & ind.valid_in_year & (ind.year == 2022)]
print(len(dfo22), "МО ДФО в 2022 г.")
dfo22[["territory_id", "gmp_pc", "wage", "invest_pc_nobudget", "density"]].describe().round(1)

229 МО ДФО в 2022 г.


,territory_id,gmp_pc,wage,invest_pc_nobudget,density
count,229.0,228.0,229.0,228.0,228.0
mean,827.3,1469455.8,76045.4,474405.5,79.9
std,594.1,3379641.5,30093.6,1623839.9,292.5
min,84.0,58031.0,35437.0,0.0,0.0
25%,343.0,339122.7,52720.0,8274.6,0.4
50%,781.0,566297.7,68099.9,46965.5,1.8
75%,872.0,1249561.8,91921.2,267936.4,7.0
max,2350.0,37376651.1,168954.8,18495943.0,3102.5


## 2. Сеть
Параметры по умолчанию — `configs/network.yaml`; любые можно переопределить. Хэш параметров идентифицирует сеть.

In [3]:
p = network.merge_params(
    network.default_params(),
    {
        "features": {"gmp_pc": 2, "invest_pc_nobudget": 1, "hhi_emp": 1, "density": 1, "old_age_share": 1},
        "geo": {"alpha": 0.8},  # 80% экономика + 20% дороги
        "sparsify": {"method": "knn_mst", "k": 7},
    },
)
nets = network.build(p, years=[2022])
net = nets[2022]
print("hash", network.config_hash(p), net.stats())
net.edges().head()

hash f204e21afe7c {'year': 2022, 'nodes': 226, 'edges': 1075, 'components': 1, 'largest_component': 226, 'isolated': 0, 'degree_min': 7, 'degree_mean': 9.513274336283185, 'degree_max': 19, 'dropped_missing': [609, 786, 1953], 'road_km: МО без связи': [311, 313, 323, 324, 618, 619, 620, 845, 854, 858, 2344, 2345, 2346, 2348, 2350]}


,source,target,weight,distance
0,84,310,0.848562,0.136220
1,84,573,0.952724,0.073977
2,84,842,0.826141,0.146907
3,84,861,0.856146,0.132478
4,84,862,0.832029,0.144150


## 3. Кластеры и индексы качества

In [4]:
X, W = net.X.to_numpy(), net.W * net.A
rows = []
for m in ["kmeans", "ward", "leiden", "spectral", "kefrin"]:
    if not clustering.available(m)[0]:
        continue
    lab = clustering.fit(X, W, {"method": m, "k": 6})
    rows.append({"метод": m, **icvi.compute_all(X, W, lab)})
pd.DataFrame(rows).round(3)

,метод,K,SW,CH,DBI,S_Dbw,AVI,AVU,ANUI,MQ
0,kmeans,6,0.296,131.294,1.127,0.502,0.736,0.488,0.541,0.553
1,ward,6,0.282,120.685,1.135,0.457,0.747,0.503,0.543,0.556
2,leiden,6,0.145,73.757,1.860,0.718,0.875,0.538,0.595,0.698
3,spectral,6,0.185,84.672,1.682,0.490,0.885,0.455,0.631,0.679
4,kefrin,6,0.149,75.374,1.729,0.732,0.837,0.516,0.585,0.628


## 4. Динамика типов: одна модель на всю панель (режим по умолчанию)

In [5]:
p_all = network.default_params()
Xp = network.features(p_all).dropna()
lab = clustering.fit_pooled(Xp, {"method": "kmeans", "k": 6}).reset_index()
th = dynamics.match_years(lab, 0.2)
dynamics.year_summary(th).round(3)

,year_from,year_to,МО в обоих годах,ARI,доля сменивших тип,новых типов,исчезло типов
0,2017,2018,223,0.561,0.202,0,0
1,2018,2019,224,0.637,0.170,0,0
2,2019,2020,205,0.519,0.215,0,0
3,2020,2021,205,0.527,0.220,0,0
4,2021,2022,224,0.545,0.219,0,0
5,2022,2023,224,0.490,0.246,0,0
6,2023,2024,200,0.550,0.220,0,0


## 5. Конвергенция ВМП на душу (базовый метод, 2013–2024, ДФО)

In [6]:
g = pd.read_parquet(PROCESSED / "gmp.parquet")
d = g[(g.method == "basic") & (g.valid_in_year | g.derived)].rename(columns={"gmp_pc_real": "value"})
d = d[d.territory_id.isin(mo[mo.is_dfo & mo.active].territory_id)][["territory_id", "year", "value"]].dropna()
sig, trend = convergence.sigma(d)
beta = convergence.beta_absolute(d, 2013, 2024)
print("σ-тренд:", {k: round(v, 4) for k, v in trend.items() if isinstance(v, float)})
print("β абсолютная:", round(beta["b"], 4), "p =", round(beta["p"], 3))

σ-тренд: {'slope': 0.0084, 'se': 0.0006, 'p': 0.0, 'sd_first': 0.8581, 'sd_last': 0.9489}
β абсолютная: -0.0048 p = 0.133
